# LSTM vs Transformer — Model Comparison

Trains both models on identical data, then compares them using bootstrap
confidence intervals so the performance difference is statistically meaningful.

**Why bootstrapping?**  Test sets per stock are ~50–80 windows.  A single AUC
score from 60 samples is noisy — bootstrapping shows whether the gap between
models is real or sampling variance.  Overlapping 95% CIs = not significant.

In [ ]:
from sentiment.log import setup_logging
setup_logging()

## 1. Load data

In [ ]:
import torch
import pandas as pd

from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalCache
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import build_dataset, make_loaders

TICKER = "AAPL"
YEAR   = 2025
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

In [ ]:
# Market data
cache = MarketDataCache()
df = cache.load(TICKER, YEAR)
if df is None or df.empty:
    raise RuntimeError(f"No market data for {TICKER} {YEAR} — run market.ipynb first")

# Fundamental data (optional — set to None to run without)
fund_cache = FundamentalCache()
fund_df = fund_cache.load(TICKER)   # None if not yet fetched

print(f"market rows : {len(df)}")
print(f"fund rows   : {len(fund_df) if fund_df is not None else 'no data'}")

In [ ]:
# Build fused dataset
technical = TechnicalFactors()
dataset = build_dataset(
    df,
    technical,
    sentiment_df=None,   # replace with SentimentPipeline output if available
    ticker=TICKER,
    fundamental_df=fund_df,
)

N_FUND = dataset["X_fundamental"].shape[1]
print(f"windows     : {len(dataset['y'])}")
print(f"n_fund      : {N_FUND}")

In [ ]:
train_loader, val_loader, test_loader, tech_scaler, fund_scaler = make_loaders(
    dataset, batch_size=32
)

## 2. Train LSTM

In [ ]:
from sentiment.model import SentimentLSTM, train_model

lstm = SentimentLSTM(n_fundamentals=N_FUND)

lstm_history = train_model(
    lstm,
    train_loader,
    val_loader,
    n_epochs=100,
    lr=1e-3,
    patience=15,
    device=DEVICE,
)
print(f"LSTM best epoch {lstm_history['best_epoch']}  val_auc={lstm_history['best_val_auc']:.4f}")

## 3. Train Transformer

In [ ]:
from sentiment.model import SentimentTransformer

transformer = SentimentTransformer(
    n_fundamentals=N_FUND,
    d_model=64,
    nhead=4,
    n_layers=2,
    dim_feedforward=128,
    dropout=0.2,
)

transformer_history = train_model(
    transformer,
    train_loader,
    val_loader,
    n_epochs=100,
    lr=1e-3,
    patience=15,
    device=DEVICE,
)
print(f"Transformer best epoch {transformer_history['best_epoch']}  val_auc={transformer_history['best_val_auc']:.4f}")

## 4. Point estimates on test set

In [ ]:
from sentiment.model import evaluate

lstm_test        = evaluate(lstm,        test_loader, device=DEVICE)
transformer_test = evaluate(transformer, test_loader, device=DEVICE)

point_df = pd.DataFrame(
    {"LSTM": lstm_test, "Transformer": transformer_test},
    index=["auc", "accuracy", "precision", "recall", "loss"],
).round(4)
display(point_df)

## 5. Bootstrap confidence intervals

1 000 resamples with replacement.  If the 95% CIs for AUC **overlap**, the
performance difference is not statistically significant at the 5% level.

In [ ]:
from sentiment.model import bootstrap_evaluate

lstm_bs        = bootstrap_evaluate(lstm,        test_loader, device=DEVICE, n_bootstrap=1000, seed=42)
transformer_bs = bootstrap_evaluate(transformer, test_loader, device=DEVICE, n_bootstrap=1000, seed=42)

print(f"Bootstrap resamples — LSTM: {lstm_bs['n_bootstrap']}, Transformer: {transformer_bs['n_bootstrap']}")
print(f"Test samples: {lstm_bs['n_samples']}")

In [ ]:
def _fmt(result, metric):
    mean = result[f"{metric}_mean"]
    lo   = result[f"{metric}_ci_low"]
    hi   = result[f"{metric}_ci_high"]
    return f"{mean:.4f}  [{lo:.4f} – {hi:.4f}]"

metrics = ["auc", "accuracy", "precision", "recall"]
rows = {
    m: {"LSTM": _fmt(lstm_bs, m), "Transformer": _fmt(transformer_bs, m)}
    for m in metrics
}
bootstrap_df = pd.DataFrame(rows).T
bootstrap_df.index.name = "metric"
print("95% bootstrap confidence intervals")
display(bootstrap_df)

## 6. Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, label in [
    (axes[0], "val_auc",  "Validation AUC"),
    (axes[1], "val_loss", "Validation Loss"),
]:
    ax.plot(lstm_history[metric],        label="LSTM",        linewidth=2)
    ax.plot(transformer_history[metric], label="Transformer", linewidth=2, linestyle="--")
    ax.axvline(lstm_history["best_epoch"] - 1,        color="C0", alpha=0.3, linestyle=":")
    ax.axvline(transformer_history["best_epoch"] - 1, color="C1", alpha=0.3, linestyle=":")
    ax.set_title(label)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()